In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Trajectories + Transition dipole moments dataframe

Here we load our dataframe consisting of:
- Raw geometries, Cartesian coordinates, Angstrom units.
    - These geometries come from $215$ trajectories (MD), $100 fs$ duration, specifically $200$ time-steps, $\Delta t = 0.5 fs$. <br>
      The geometries are ordered consecutively, scanning trajectory one after another.
- Raw transition dipole moments $\mu_{10} = <\Psi_1|\hat{\mu}|\Psi_0>$ between ground ($S_0$) and first excited ($S_1$) state, at each geometry. <br> Indeed one has $\mu_{10}(t) \equiv \mu_{10}(R_j(t))$ with $R_j(t)$ being the $j$-th nuclear geometry.
- Phase - corrected transition dipole moments, according to the procedure described afterwards. 

In [2]:
data = pd.read_csv("trajs-dataframe-pc.csv")
data

,TRAJ,Time,x1,y1,z1,x2,y2,z2,x3,y3,...,z23,x24,y24,z24,mu_x,mu_y,mu_z,mu_x_pc,mu_y_pc,mu_z_pc
0,1,0.5,-0.903948,2.567011,0.066012,1.375391,1.360069,-0.122150,0.848979,0.165433,...,0.206331,-4.208275,-0.299556,-0.045503,-1.298255,0.060673,-0.002043,-1.298255,0.060673,-0.002043
1,1,1.0,-0.903835,2.562395,0.065014,1.380791,1.361061,-0.124760,0.841877,0.155487,...,0.193118,-4.222286,-0.319823,-0.051924,1.330744,-0.063285,0.004413,-1.330744,0.063285,-0.004413
2,1,1.5,-0.903613,2.557535,0.064056,1.385793,1.362369,-0.127433,0.834433,0.145367,...,0.180662,-4.265446,-0.344548,-0.060105,-1.377839,0.065160,-0.006069,-1.377839,0.065160,-0.006069
3,1,2.0,-0.903245,2.552448,0.063149,1.390274,1.363923,-0.130158,0.826782,0.135235,...,0.169053,-4.328030,-0.371130,-0.069135,1.431230,-0.067969,0.006861,-1.431230,0.067969,-0.006861
4,1,2.5,-0.902709,2.547157,0.062301,1.394119,1.365663,-0.132925,0.819068,0.125243,...,0.158393,-4.399876,-0.396572,-0.077975,1.484248,-0.072878,0.006718,-1.484248,0.072878,-0.006718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42995,215,98.0,-0.934367,2.541373,0.038740,1.475924,1.438822,0.065239,0.778559,0.125982,...,0.018664,-4.513288,-0.037115,-0.031534,-1.884159,0.072271,-0.012103,-1.884159,0.072271,-0.012103
42996,215,98.5,-0.930888,2.543931,0.038357,1.476764,1.435126,0.065179,0.778242,0.132994,...,0.006837,-4.498413,-0.056363,-0.037674,1.861778,-0.057815,0.010398,-1.861778,0.057815,-0.010398
42997,215,99.0,-0.927550,2.546153,0.037961,1.477164,1.430590,0.064979,0.778772,0.140455,...,-0.004734,-4.487472,-0.077722,-0.043962,-1.829654,0.037963,-0.008759,-1.829654,0.037963,-0.008759
42998,215,99.5,-0.924355,2.548018,0.037551,1.477064,1.425246,0.064641,0.780161,0.148247,...,-0.015892,-4.482751,-0.100842,-0.050284,1.787543,-0.012123,0.007323,-1.787543,0.012123,-0.007323


# Tdm phase-correction procedure

We extract columns containing $\mu_x, \mu_y, \mu_z$ and work with Numpy arrays from now on.

In [3]:
transition = data[["mu_x", "mu_y", "mu_z"]].to_numpy()

# x component
x = transition[:, 0]

# y component
y = transition[:, 1]

# z component
z = transition[:, 2]

transition.shape, x.shape, y.shape, z.shape

((43000, 3), (43000,), (43000,), (43000,))

We reshape the transition dipoles so that they are 'tagged' by trajectory.

In [4]:
# tag tdm (or tdm component) by trajectory

n_trajs = 215

tdm_trajs = transition.reshape(n_trajs, 200, 3)

x_trajs = x.reshape(n_trajs, -1)
y_trajs = y.reshape(n_trajs, -1)
z_trajs = z.reshape(n_trajs, -1)

tdm_trajs.shape, x_trajs.shape, y_trajs.shape, z_trajs.shape

((215, 200, 3), (215, 200), (215, 200), (215, 200))

The phase correction procedure.

In [5]:
# phase-correct the full dataset with a reference tdm 
# when the angle between the dipoles surpasses cos_threshold, flip sign

def phase_correct_cos(tdm_array, tdm_ref, cos_threshold):

    ref_norm = np.linalg.norm(tdm_ref) # compute norm of the reference
    
    for i, traj in enumerate(tdm_array):
        for j, tdm in enumerate(traj):
            # compute cos from dot product
            cos_theta = np.dot(tdm, tdm_ref) / (np.linalg.norm(tdm)*ref_norm)
        
            if cos_theta < cos_threshold:
                tdm_array[i, j] = tdm * (-1)

    return tdm_array

We apply the correction selecting the 1st transition dipole of the 1st trajectory as reference tdm.

In [6]:
# the cos - phase - corrected full dataset
# threshold is angle pi/2, i.e. all cos_theta < 0 tdm get corrected

tdm_ref = tdm_trajs[0, 0]
cos_threshold = 0.0
tdm_corrected_cos = phase_correct_cos(tdm_trajs, tdm_ref, cos_threshold)

Finally, here are the components $\mu_x^{pc}, \mu_y^{pc}, \mu_z^{pc}$we added (last 3 columns) to the dataframe.

In [7]:
# phase-corrected components

x_pc = tdm_corrected_cos[:, :, 0]
y_pc = tdm_corrected_cos[:, :, 1]
z_pc = tdm_corrected_cos[:, :, 2]

x_pc.shape, y_pc.shape, z_pc.shape

((215, 200), (215, 200), (215, 200))

# Geometries 

## Translational invariance

Given our initial geometries dataset, we would like to encode one essential physical symmetry feature, namely **translational invariance**.<br>
Indeed, two geometries differing only by a translation (equally applied to each atom) in our Cartesian reference frame, should be seen as equal inputs by the ML model. <br>
Therefore, the ML model is expected to produce the same output $\mu^{pred}$.

We extract our geometries array from the previous dataframe, ranging from **x1** to **z24**.

In [9]:
# number of atoms in HBQ molecule
nat = 24
final_column = 2 + (nat * 3)

arr = data.iloc[:, 2:final_column].to_numpy()
geoms = arr.reshape(-1, 24, 3)
geoms.shape

(43000, 24, 3)

For our procedure, we need the nuclear masses list of HBQ molecule, which we picked from the **/TRAJ1/geom** file, <br>
from NewtonX **TRAJECTORIES** folder.

In [10]:
masses = np.loadtxt("geom", usecols=(-1,))
masses.shape

(24,)

Masses are in A.M.U., but any unit system would be good, because they simplify out in our procedure.

In [11]:
masses

array([15.99491464, 14.00307401, 12.        , 12.        , 12.        ,
       12.        , 12.        , 12.        , 12.        , 12.        ,
       12.        , 12.        , 12.        , 12.        , 12.        ,
        1.00782504,  1.00782504,  1.00782504,  1.00782504,  1.00782504,
        1.00782504,  1.00782504,  1.00782504,  1.00782504])

In the following, we encode translational invariance according to this:
1. Compute C.O.M. (center of mass) for each geometry, store in array.
2. Express each geometry in its C.O.M. coordinates, i.e. the C.O.M. is the new origin. 

In [12]:
# Array with centres of mass for all geoms

def allcom(geoms):
    coms = np.zeros((len(geoms), 3))
    for i in range(len(geoms)):
        num, den = 0, 0
        for j in range(geoms.shape[1]):
            num = num + (masses[j] * geoms[i, j])
            den = den + masses[j]
        coms[i] = num / den
    return coms

allcom(geoms).shape

(43000, 3)

In [13]:
# Translate geometries

geoms_transl = np.zeros(geoms.shape)

def translateg(geoms, geoms_com):
    for i in range(len(geoms)):
        for j in range(geoms.shape[1]):
            geoms_transl[i, j] = geoms[i, j] - geoms_com[i]
    return geoms_transl

translateg(geoms, allcom(geoms)).shape

(43000, 24, 3)

Finally, here is our geometries dataset encoding translational invariance, <br>
conveniently stored in a Numpy array.

In [14]:
geomst = translateg(geoms, allcom(geoms))